# ipyleaflet GNSS Station Popup Demo

## Import Packages

In [1]:
from pathlib import Path
import sys

local_packages = Path("ipyleaflet_packages").resolve()
if local_packages.exists():
    sys.path.insert(0, str(local_packages))

from ipyleaflet import Map, CircleMarker, Polyline, WidgetControl, basemaps
import ipywidgets as wg
import matplotlib.pyplot as plt
import numpy as np
from datetime import datetime

## Sample Station Data

In [2]:
stations = {
    ("P040", "NGF"): {
        "location": (34.0556, -118.2468),
        "velocity": (-4.96, -14.34, -0.32),
        "velsig": (0.004, 0.003, 0.015),
        "height_m": 89.3,
        "note": "Los Angeles area sample station",
    },
    ("P123", "UNR"): {
        "location": (36.1699, -115.1398),
        "velocity": (-3.21, -11.82, 0.64),
        "velsig": (0.018, 0.016, 0.052),
        "height_m": 612.7,
        "note": "Las Vegas area sample station",
    },
    ("AB01", "JPL"): {
        "location": (37.7749, -122.4194),
        "velocity": (-5.42, -18.67, 1.12),
        "velsig": (0.011, 0.010, 0.040),
        "height_m": 18.2,
        "note": "San Francisco area sample station",
    },
}

## 3. Shared widgets

In [3]:
# Initize map
map_widget = None
selected_station = {"siteid": None, "org": None}

# UI elements for when you're selecting and displaying station info (we will change this later probably)
selected_html = wg.HTML("<b>Selected station:</b> none")
ts_sites = wg.SelectMultiple(options=[], description="TS list:", rows=3, layout=wg.Layout(width="245px"))
copy_box = wg.Textarea(
    value="Click a station, then use Prepare Copy Text.",
    description="Copy text:",
    layout=wg.Layout(width="245px", height="85px"),
)
plot_output = wg.Output()

# Helper functions for station info and plotting

def station_label(siteid: str, org: str) -> str:
    """
    Return the display label used by the time-series selector.
    """
    return f"{siteid} ({org})"

def station_summary(siteid: str, org: str) -> str:
    """
    Return a copy-friendly summary for a station record.

    OG notebook station metadata (might change depending on professor's call):
    location, height, NEU velocity, and NEU velocity uncertainty.
    """

    info = stations[(siteid, org)]
    vn, ve, vu = info["velocity"]
    sn, se, su = info["velsig"]
    lat, lon = info["location"]
    return (
        f"Site: {siteid}\n"
        f"Source: {org}\n"
        f"Lat/Lon: {lat:.5f}, {lon:.5f}\n"
        f"Height: {info['height_m']:.1f} m\n"
        f"Velocity NEU: {vn:.2f}, {ve:.2f}, {vu:.2f} mm/yr\n"
        f"Sigma NEU: {sn:.3f}, {se:.3f}, {su:.3f} mm/yr\n"
        f"Note: {info['note']}"
    )

def select_station(siteid: str, org: str) -> None:
    """
    Set the active station and recenter the map on it.
    """
    selected_station["siteid"] = siteid
    selected_station["org"] = org
    info = stations[(siteid, org)]
    vn, ve, vu = info["velocity"]
    selected_html.value = (
        f"<b>Selected station:</b> {siteid} ({org})<br>"
        f"Velocity NEU: {vn:.2f}, {ve:.2f}, {vu:.2f} mm/yr"
    )
    if map_widget is not None:
        map_widget.center = info["location"]
        map_widget.zoom = max(map_widget.zoom, 8)

def add_to_timeseries(siteid: str, org: str) -> None:
    """
    Add a station to the time-series selection widget.
    """
    label = station_label(siteid, org)
    if label not in ts_sites.options:
        ts_sites.options = tuple(list(ts_sites.options) + [label])
    ts_sites.value = tuple([label])

def prepare_copy_text(siteid: str, org: str) -> None:
    """
    Fill the copy box with station metadata.
    """
    copy_box.value = station_summary(siteid, org)

def plot_mock_timeseries(siteid: str, org: str) -> None:
    """
    Render fake time-series data for a station. This was just for the demo.

    In the original notebook, this is where the Read/plot time-series data code would go,
    but since I didn't actually put any in (lol) this is just a random placeholder that follows the velocity
    trend with some noise. Don't focus too much on it for now, we're gonna take it out anyway.
    """
    info = stations[(siteid, org)]
    years = np.linspace(2018, 2026, 120)
    t = years - years[0]
    vn, ve, vu = info["velocity"]
    rng = np.random.default_rng(abs(hash((siteid, org))) % 2**32)

    north = vn * t + rng.normal(0, 1.5, len(t))
    east = ve * t + rng.normal(0, 1.5, len(t))
    up = vu * t + 3 * np.sin(2 * np.pi * t) + rng.normal(0, 4.0, len(t))

    with plot_output:
        plot_output.clear_output(wait=True)
        fig, axes = plt.subplots(3, 1, figsize=(8, 5), sharex=True)
        for ax, values, title in zip(axes, [north, east, up], ["North", "East", "Up"]):
            ax.plot(years, values, linewidth=1.2)
            ax.set_ylabel(f"{title} mm")
            ax.grid(True, alpha=0.25)
        axes[-1].set_xlabel("Year")
        fig.suptitle(f"Mock time series for {siteid} ({org})")
        plt.tight_layout()
        plt.show()

def plot_selected(button: wg.Button | None = None) -> None:
    """
    Plot the first station selected in the time-series widget.
    """
    if not ts_sites.value:
        with plot_output:
            plot_output.clear_output(wait=True)
            print("Select or add a station first.")
        return
    label = ts_sites.value[0]
    siteid, org = label.replace(")", "").split(" (")
    plot_mock_timeseries(siteid, org)

plot_selected_button = wg.Button(description="Plot Selected", button_style="info", layout=wg.Layout(width="120px"))
plot_selected_button.on_click(plot_selected)

copy_section = wg.Accordion(children=[copy_box])
copy_section.set_title(0, "Copy text")
copy_section.selected_index = None

side_panel = wg.VBox([
    selected_html,
    ts_sites,
    wg.HBox([plot_selected_button]),
    copy_section,
], layout=wg.Layout(width="265px"))

## Popups

In [4]:
def make_popup_child(siteid: str, org: str) -> wg.VBox:
    """
    Build the live widget popup for a station dot.
    Then assign the returned widget directly to the ipyleaflet layer's popup.
    """
    info = stations[(siteid, org)]
    vn, ve, vu = info["velocity"]

    title = wg.HTML(
        f"<b>{siteid}</b> ({org})<br>"
        f"NEU velocity: {vn:.2f}, {ve:.2f}, {vu:.2f} mm/yr"
    )
    select_button = wg.Button(description="Select", layout=wg.Layout(width="95px"))
    add_button = wg.Button(description="Add to TS", layout=wg.Layout(width="110px"))
    copy_button = wg.Button(description="Prepare Copy Text", layout=wg.Layout(width="170px"))
    plot_button = wg.Button(description="Plot Now", button_style="success")

    def on_select(button: wg.Button) -> None:
        """
        Select this station and zoom the map to it.
        (I want to incorporate the dynamic zooming in the future,
        but for now it just centers the map on the station and zooms in.)
        """
        select_station(siteid, org)

    def on_add(button: wg.Button) -> None:
        """
        Add this station to the time-series list.
        """
        select_station(siteid, org)
        add_to_timeseries(siteid, org)

    def on_copy(button: wg.Button) -> None:
        """
        Copy this station's metadata into the text box.
        """
        select_station(siteid, org)
        prepare_copy_text(siteid, org)

    def on_plot(button: wg.Button) -> None:
        """
        Add this station and plot its demo time series immediately.
        """
        select_station(siteid, org)
        add_to_timeseries(siteid, org)
        plot_mock_timeseries(siteid, org)

    select_button.on_click(on_select)
    add_button.on_click(on_add)
    copy_button.on_click(on_copy)
    plot_button.on_click(on_plot)

    return wg.VBox([
        title,
        wg.HBox([select_button, add_button]),
        copy_button,
        plot_button,
    ])

## Map Display

In [5]:
m = Map(
    center=(36.0, -119.0),
    zoom=5,
    basemap=basemaps.OpenStreetMap.Mapnik,
    scroll_wheel_zoom=True,
    layout=wg.Layout(width="100%", height="560px"),
)
map_widget = m

velocity_scale = wg.FloatSlider(
    value=10,
    min=2,
    max=20,
    step=1,
    description="km/mm/yr",
    readout_format=".0f",
    layout=wg.Layout(width="260px"),
)
guide_velocity = wg.FloatSlider(
    value=10,
    min=5,
    max=20,
    step=5,
    description="guide mm/yr",
    readout_format=".0f",
    layout=wg.Layout(width="260px"),
)
scale_status = wg.HTML()
guide_bar = wg.HTML()
vector_layers = []

def offset_location(lat: float, lon: float, north_km: float = 0, east_km: float = 0) -> tuple[float, float]:
    """
    Return an approximate lat/lon after applying local north/east offsets.

    (this is only good for short distances, we are also revamping this probably).
    """
    km_per_degree_lat = 111.32
    km_per_degree_lon = 111.32 * np.cos(np.deg2rad(lat))
    if abs(km_per_degree_lon) < 1e-6:
        km_per_degree_lon = 1e-6
    return (
        lat + north_km / km_per_degree_lat,
        lon + east_km / km_per_degree_lon,
    )

def remove_layers(layers: list[Polyline]) -> None:
    """
    Remove temporary vector layers from the map.
    """
    for layer in list(layers):
        try:
            m.remove(layer)
        except Exception:
            pass
    layers.clear()

def refresh_velocity_vectors(event: object | None = None) -> None:
    """
    Redraw station velocity vectors using the current scale setting.
    """
    remove_layers(vector_layers)

    scale = velocity_scale.value
    for (siteid, org), info in stations.items():
        lat, lon = info["location"]
        vn, ve, vu = info["velocity"]
        end = offset_location(lat, lon, north_km=vn * scale, east_km=ve * scale)
        color = "#2166ac" if vu >= 0 else "#b2182b"
        vector = Polyline(
            locations=[info["location"], end],
            color=color,
            weight=3,
            opacity=0.85,
        )
        vector.popup = wg.HTML(
            f"<b>{siteid} ({org})</b><br>"
            f"Horizontal velocity: N {vn:.2f}, E {ve:.2f} mm/yr<br>"
            f"Vertical velocity: {vu:.2f} mm/yr"
        )
        m.add(vector)
        vector_layers.append(vector)

    refresh_fixed_guide()

def refresh_fixed_guide(event: object | None = None) -> None:
    """
    Update the fixed vector guide for the current zoom and scale.
    """
    scale = velocity_scale.value
    guide_km = guide_velocity.value * scale
    center_lat = m.center[0] if m.center else 0
    meters_per_pixel = 156543.03392 * np.cos(np.deg2rad(center_lat)) / (2 ** m.zoom)
    raw_px = guide_km * 1000 / max(meters_per_pixel, 1e-6)
    bar_px = min(max(raw_px, 18), 220)
    note = "" if raw_px <= 220 else "<br><span style='color:#666'>bar shortened to fit panel</span>"
    guide_bar.value = (
        "<div style='font-size:12px; line-height:1.25'>"
        f"<b>{guide_velocity.value:.0f} mm/yr</b> vector guide<br>"
        f"<div style='width:{bar_px:.0f}px; height:6px; background:#111; margin:6px 0 3px 0'></div>"
        f"{guide_km:.0f} km at current scale{note}"
        "</div>"
    )
    scale_status.value = (
        f"<span style='font-size:12px'>Station vectors: {scale:.0f} km per mm/yr</span>"
    )

velocity_scale.observe(refresh_velocity_vectors, names="value")
guide_velocity.observe(refresh_velocity_vectors, names="value")
m.observe(refresh_fixed_guide, names="zoom")
m.observe(refresh_fixed_guide, names="center")

for (siteid, org), info in stations.items():
    dot = CircleMarker(
        location=info["location"],
        radius=6,
        color="black",
        fill_color="#2b83ba",
        fill_opacity=0.9,
        weight=1,
        name=station_label(siteid, org),
    )
    dot.popup = make_popup_child(siteid, org)
    m.add(dot)

vector_control = wg.VBox([
    wg.HTML("<b>Velocity vector scale</b>"),
    guide_bar,
    velocity_scale,
    guide_velocity,
    scale_status,
], layout=wg.Layout(width="285px"))

m.add(WidgetControl(widget=side_panel, position="bottomright"))
m.add(WidgetControl(widget=vector_control, position="bottomleft"))
refresh_velocity_vectors()

display(wg.VBox([m, plot_output]))